In [6]:
from __future__ import annotations

from pathlib import Path
from typing import Sequence, Union, Optional
import re

import numpy as np
import tifffile as tf

from ome_types import from_xml, to_xml
from ome_types.model import OME, Channel, TiffData

ROUND_RE = re.compile(r"_R\d{3}_")  # _R000_, _R012_, etc.


def infer_channel_name(path: Union[str, Path]) -> str:
    """
    Infer channel/marker name from CellDIVE-ish filenames.

    Examples
    --------
    SLIDE-0272_1.0.4_R000_DAPI__FINAL_F.ome.tif -> DAPI
    Cy3_NaK-ATPase-555.ome.tif                 -> Cy3_NaK-ATPase-555
    """
    p = Path(path)
    name = p.name
    name = re.sub(r"\.ome\.tif(f)?$", "", name, flags=re.IGNORECASE)

    m = ROUND_RE.search(name)
    if m:
        after = name[m.end():]
        after = after.split("__", 1)[0]
        after = after.split("_FINAL", 1)[0]
        after = after.split("_Final", 1)[0]
        ch = after.strip("_- ")
    else:
        ch = name.strip("_- ")

    ch = re.sub(r"__+.*$", "", ch)
    ch = re.sub(r"(_FINAL.*)$", "", ch, flags=re.IGNORECASE)
    return ch.strip("_- ")


def _read_level0_yx(path: Union[str, Path]) -> np.ndarray:
    path = Path(path)
    with tf.TiffFile(path) as t:
        s0 = t.series[0]
        a = s0.asarray(level=0)  # full-res only
        if a.ndim == 3 and a.shape[0] == 1:
            a = a[0]
        if a.ndim != 2:
            raise ValueError(f"{path}: expected single-channel 2D (YX). Got {a.shape}")
        return a


def _get_ome_xml(path: Union[str, Path]) -> str:
    path = Path(path)
    with tf.TiffFile(path) as t:
        ome = t.ome_metadata
        if ome is None:
            raise ValueError(f"{path} missing OME-XML (t.ome_metadata is None).")
        return ome


def _extract_single_channel(ome_xml: str) -> Channel:
    ome: OME = from_xml(ome_xml)
    if not ome.images or ome.images[0].pixels is None:
        raise ValueError("OME-XML missing Image/Pixels.")
    px = ome.images[0].pixels
    chs = list(px.channels or [])
    if len(chs) != 1:
        raise ValueError(f"Expected exactly 1 channel in input OME-XML, found {len(chs)}.")
    return chs[0]


def _downsample2x_mean(a: np.ndarray) -> np.ndarray:
    """2x downsample by 2x2 mean. Crops last row/col if Y or X is odd."""
    y, x = a.shape
    y2 = y - (y % 2)
    x2 = x - (x % 2)
    a = a[:y2, :x2]  # ensure even
    a_f = a.astype(np.float32, copy=False)
    return (a_f[0::2, 0::2] + a_f[1::2, 0::2] + a_f[0::2, 1::2] + a_f[1::2, 1::2]) / 4.0


def _build_pyramid(level0: np.ndarray, min_size: int = 1024) -> list[np.ndarray]:
    pyr = [level0]
    while True:
        y, x = pyr[-1].shape
        if min(y, x) <= min_size:
            break
        nxt = _downsample2x_mean(pyr[-1]).astype(level0.dtype)
        pyr.append(nxt)
    return pyr


def merge_single_channel_ometiffs_preserve_metadata_streaming(
    inputs: Sequence[Union[str, Path]],
    output: Union[str, Path],
    *,
    tile: int = 512,
    compression: str = "lzw",
    bigtiff: bool = True,
    channel_names: Optional[Sequence[str]] = None,
    infer_names_from_filenames: bool = True,
    enforce_same_physical_size: bool = True,
    normalize_dimension_order_xyzct: bool = True,
    make_pyramid: bool = False,
    pyramid_min_size: int = 1024,
) -> Path:
    """
    Merge per-channel OME-TIFFs into one multi-channel OME-TIFF while preserving OME-XML metadata.

    - Base metadata from first file (Instrument/Objective/etc preserved)
    - Channel metadata copied from each file's single <Channel>
    - Channel names optionally inferred from filenames
    - Writes one IFD/page per channel; optionally writes pyramid SubIFDs per channel
    - Pixel IO is one channel at a time

    Pyramid note:
      If make_pyramid=True, pyramids are regenerated (SubIFDs). Existing pyramids are not reused.

    Returns output path.
    """
    if not inputs:
        raise ValueError("inputs must be non-empty")
    inputs = [Path(p) for p in inputs]
    output = Path(output)

    if channel_names is not None and len(channel_names) != len(inputs):
        raise ValueError("channel_names must have same length as inputs")

    if channel_names is None and infer_names_from_filenames:
        channel_names = [infer_channel_name(p) for p in inputs]

    # ---- OME metadata merge (small) ----
    ome_xml0 = _get_ome_xml(inputs[0])
    base: OME = from_xml(ome_xml0)
    img0 = base.images[0]
    px0 = img0.pixels
    if px0 is None:
        raise ValueError("Base OME-XML missing Pixels.")

    base_pxX, base_pxY = px0.physical_size_x, px0.physical_size_y
    base_unitX, base_unitY = px0.physical_size_x_unit, px0.physical_size_y_unit

    channels: list[Channel] = []
    for i, p in enumerate(inputs):
        xml = ome_xml0 if i == 0 else _get_ome_xml(p)
        ch = _extract_single_channel(xml)

        if channel_names is not None:
            ch.name = channel_names[i]

        if enforce_same_physical_size and i > 0:
            o = from_xml(xml)
            pxi = o.images[0].pixels
            if pxi is None:
                raise ValueError(f"{p} missing Pixels in OME-XML.")
            if (pxi.physical_size_x != base_pxX) or (pxi.physical_size_y != base_pxY) or \
               (pxi.physical_size_x_unit != base_unitX) or (pxi.physical_size_y_unit != base_unitY):
                raise ValueError(
                    f"Physical pixel size mismatch in {p} vs {inputs[0]}.\n"
                    f"Base: ({base_pxX} {base_unitX}, {base_pxY} {base_unitY})\n"
                    f"This:  ({pxi.physical_size_x} {pxi.physical_size_x_unit}, "
                    f"{pxi.physical_size_y} {pxi.physical_size_y_unit})"
                )

        channels.append(ch)

    px0.size_c = len(channels)
    px0.channels = channels
    px0.size_z = px0.size_z or 1
    px0.size_t = px0.size_t or 1

    if normalize_dimension_order_xyzct:
        try:
            from ome_types.model import Pixels_DimensionOrder
            px0.dimension_order = Pixels_DimensionOrder.XYZCT
        except Exception:
            pass

    # Map output IFD/page -> C index (Z=0, T=0). (Pyramids are SubIFDs off each IFD; mapping remains valid.)
    px0.tiff_data_blocks = [
        TiffData(ifd=c, first_z=0, first_c=c, first_t=0, plane_count=1)
        for c in range(len(channels))
    ]

    merged_ome_xml = to_xml(base)

    # ---- Pixel IO: write one channel at a time ----
    first = _read_level0_yx(inputs[0])
    yx0 = first.shape
    dt0 = first.dtype

    with tf.TiffWriter(output, bigtiff=bigtiff) as tw:
        # Channel 0
        if not make_pyramid:
            tw.write(
                first,
                tile=(tile, tile),
                compression=compression,
                photometric="minisblack",
                description=merged_ome_xml,  # OME-XML on first IFD
                metadata=None,
            )
        else:
            pyr = _build_pyramid(first, min_size=pyramid_min_size)
            tw.write(
                pyr[0],
                tile=(tile, tile),
                compression=compression,
                photometric="minisblack",
                subifds=len(pyr) - 1,
                description=merged_ome_xml,
                metadata=None,
            )
            for lvl in pyr[1:]:
                tw.write(
                    lvl,
                    tile=(tile, tile),
                    compression=compression,
                    photometric="minisblack",
                    subfiletype=1,
                    metadata=None,
                )

        # Remaining channels
        for p in inputs[1:]:
            a = _read_level0_yx(p)
            if a.shape != yx0:
                raise ValueError(f"Shape mismatch: {p} has {a.shape}, expected {yx0}")
            if a.dtype != dt0:
                raise ValueError(f"Dtype mismatch: {p} has {a.dtype}, expected {dt0}")

            if not make_pyramid:
                tw.write(
                    a,
                    tile=(tile, tile),
                    compression=compression,
                    photometric="minisblack",
                    metadata=None,
                )
            else:
                pyr = _build_pyramid(a, min_size=pyramid_min_size)
                tw.write(
                    pyr[0],
                    tile=(tile, tile),
                    compression=compression,
                    photometric="minisblack",
                    subifds=len(pyr) - 1,
                    metadata=None,
                )
                for lvl in pyr[1:]:
                    tw.write(
                        lvl,
                        tile=(tile, tile),
                        compression=compression,
                        photometric="minisblack",
                        subfiletype=1,
                        metadata=None,
                    )

    return output

In [7]:
#d = Path("/path/to/channels")
#inputs = sorted(d.glob("SlideA_*.ome.tif"))  # adjust pattern
inputs = [
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0272/SLIDE-0272_1.0.4_R000_DAPI__FINAL_F.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0272/Cy3_NaK-ATPase-555.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0272/Cy5_CD45-CST-AF647.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0272/Cy7_PanCK-ae1-ae3-750.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0272/Cy7_Podoplanin-750.ome.tif",
]

out = merge_single_channel_ometiffs_preserve_metadata_streaming(
    inputs=inputs,
    output=r"/data1/lowes/ratnayn/Data/M10_admix/SLIDE-0272_segment_merge.ome.tif",
    compression="lzw",
    tile=512,
    make_pyramid = False
)
print(out)

/data1/lowes/ratnayn/Data/M10_admix/SLIDE-0272_segment_merge.ome.tif


In [8]:
#d = Path("/path/to/channels")
#inputs = sorted(d.glob("SlideA_*.ome.tif"))  # adjust pattern
inputs = [
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0273/SLIDE-0273_1.0.4_R000_DAPI__FINAL_F.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0273/Cy3_NaK-ATPase-555.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0273/Cy5_CD45-CST-AF647.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0273/Cy7_PanCK-ae1-ae3-750.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0273/Cy7_Podoplanin-750.ome.tif",
]

out = merge_single_channel_ometiffs_preserve_metadata_streaming(
    inputs=inputs,
    output=r"/data1/lowes/ratnayn/Data/M10_admix/SLIDE-0273_segment_merge.ome.tif",
    compression="lzw",
    tile=512,
    make_pyramid = False
)
print(out)

/data1/lowes/ratnayn/Data/M10_admix/SLIDE-0273_segment_merge.ome.tif


In [9]:
#d = Path("/path/to/channels")
#inputs = sorted(d.glob("SlideA_*.ome.tif"))  # adjust pattern
inputs = [
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0275/SLIDE-0275_1.0.4_R000_DAPI__FINAL_F.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0275/Cy3_NaK-ATPase-555.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0275/Cy5_CD45-D3F8Q-555.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0275/Cy7_PanCK-ae1-ae3-750.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0275/Cy7_Podoplanin-750.ome.tif",
]

out = merge_single_channel_ometiffs_preserve_metadata_streaming(
    inputs=inputs,
    output=r"/data1/lowes/ratnayn/Data/M10_admix/SLIDE-0275_segment_merge_pyramid.ome.tif",
    compression="lzw",
    tile=512,
    make_pyramid = False
)
print(out)

/data1/lowes/ratnayn/Data/M10_admix/SLIDE-0275_segment_merge_pyramid.ome.tif


In [10]:
#d = Path("/path/to/channels")
#inputs = sorted(d.glob("SlideA_*.ome.tif"))  # adjust pattern
inputs = [
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0276/SLIDE-0276_1.0.4_R000_DAPI__FINAL_F.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0276/Cy3_NaK-ATPase-555.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0276/Cy5_CD45-D3F8Q-555.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0276/Cy7_PanCK-ae1-ae3-750.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0276/Cy7_Podoplanin-750.ome.tif",
]

out = merge_single_channel_ometiffs_preserve_metadata_streaming(
    inputs=inputs,
    output=r"/data1/lowes/ratnayn/Data/M10_admix/SLIDE-0276_segment_merge.ome.tif",
    compression="lzw",
    tile=512,
    make_pyramid = False
)
print(out)

/data1/lowes/ratnayn/Data/M10_admix/SLIDE-0276_segment_merge.ome.tif


In [11]:
#d = Path("/path/to/channels")
#inputs = sorted(d.glob("SlideA_*.ome.tif"))  # adjust pattern
inputs = [
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0277/SLIDE-0277_1.0.4_R000_DAPI__FINAL_F.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0277/Cy3_NaK-ATPase-555.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0277/Cy5_CD45-D3F8Q-555.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0277/Cy7_PanCK-ae1-ae3-750.ome.tif",
    r"/data1/lowes/dmh/PDAC/data/PDAC_ADMX_image_100825/image_data/SLIDE-0277/Cy7_Podoplanin-750.ome.tif",
]

out = merge_single_channel_ometiffs_preserve_metadata_streaming(
    inputs=inputs,
    output=r"/data1/lowes/ratnayn/Data/M10_admix/SLIDE-0277_segment_merge.ome.tif",
    compression="lzw",
    tile=512,
    make_pyramid = False
)
print(out)

/data1/lowes/ratnayn/Data/M10_admix/SLIDE-0277_segment_merge.ome.tif


In [3]:
import tifffile as tf
from ome_types import from_xml

path = "/data1/lowes/ratnayn/Data/M10_admix/SLIDE-0272_segment_merge_pyramid.ome.tif"

with tf.TiffFile(path) as t:
    ome = from_xml(t.ome_metadata)

px = ome.images[0].pixels
print("PhysicalSizeX:", px.physical_size_x, px.physical_size_x_unit)
print("PhysicalSizeY:", px.physical_size_y, px.physical_size_y_unit)
print("SizeX, SizeY:", px.size_x, px.size_y)
print("DimensionOrder:", px.dimension_order)
print("SizeC:", px.size_c, "SizeZ:", px.size_z, "SizeT:", px.size_t)

PhysicalSizeX: 0.325002437518281 UnitsLength.MICROMETER
PhysicalSizeY: 0.325002437518281 UnitsLength.MICROMETER
SizeX, SizeY: 59212 60899
DimensionOrder: Pixels_DimensionOrder.XYZCT
SizeC: 5 SizeZ: 1 SizeT: 1
